In [6]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
import sys
import os
from pathlib import Path

# Subimos un nivel y entramos en 'scripts'
ruta_scripts = str(Path(os.getcwd()).parent / "scripts")
if ruta_scripts not in sys.path:
    sys.path.append(ruta_scripts)

# Ahora ya puedes importar tu clase
try:
    from isolation_cleaner import IsolationCleaner
    print("✅ Clase IsolationCleaner cargada correctamente.")
except ImportError as e:
    print(f"❌ Error al importar: {e}")

import matplotlib.pyplot as plt
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from pathlib import Path

def graficar_percentiles_limpios_clase_custom(X, y, k, percentil_dist=70, porcentaje_limpieza=1.0, test_size=0.2, random_state=42, nombre_dataset="", guardar_en=None, 
nombre_configuracion_dataset={}):
    """
    Replica el flujo exacto de preprocesamiento y grafica curvas OVA ordenando la leyenda por valor de radio.
    """
    # 1. SPLIT TRAIN/TEST (Estratificado)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    print(f"\n--- TAMAÑO DE LOS CONJUNTOS ({nombre_dataset.upper()}) ---")
    print(f"Total de datos originales: {len(X)}")
    print(f"Tamaño Train: {X_train.shape[0]} muestras ({100*(1-test_size):.0f}%)")
    print(f"Tamaño Test:  {X_test.shape[0]} muestras ({100*test_size:.0f}%)")

    etiquetas, conteos = np.unique(y_train, return_counts=True)
    
    print(f"\n--- DISTRIBUCIÓN DE CLASES EN TRAIN (80% de los datos) ---")
    for etiqueta, conteo in zip(etiquetas, conteos):
        print(f"Clase {etiqueta}: {conteo} instancias")
    print(f"Total Train: {len(y_train)}")
    # ------------------------------------------

    # 2. ESCALADO ROBUSTO
    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    # 3. LIMPIEZA CON IsolationCleaner
    X_train_final, y_train_final, _ = IsolationCleaner.limpiarOutliers(
        X=X_train_scaled,
        y=y_train,
        percentil_umbral=float(porcentaje_limpieza),
        random_state=random_state,
        verbose=True
    )

    # 4. ENTRENAR KNN GLOBAL
    nn = NearestNeighbors(n_neighbors=k + 1, metric="euclidean")
    nn.fit(X_train_final)
    
    clases = np.unique(y_train_final)
    cmap = plt.get_cmap('tab10')
    plt.figure(figsize=(12, 7))
    
    # Listas para almacenar handles y etiquetas para ordenar la leyenda
    legend_info = []
    radios_calculados = {}

    # etiquetas a graficar
    etiqueta_a_graficar = {
        'shuttle' : [1,4,5,3,2],
    }

    # Umbrales manuales proporcionados para SHUTTLE
    umbrales_manuales = {
        1: 1.366, 2: 22.34, 3: 8.502, 4: 1.958, 
        5: 4.118, 6: 2020.008, 7: 2282.093
    }

    for i, etiqueta_clase in enumerate(clases):

        # if nombre_dataset=='shuttle' and etiqueta_clase not in etiqueta_a_graficar['shuttle']:
        #     continue

        X_pos = X_train_final[y_train_final == etiqueta_clase]
        if len(X_pos) < 2: continue
        
        # Cálculo de distancias (Lógica OVA)
        distancias, _ = nn.kneighbors(X_pos)
        distancias_vector = np.sort(distancias[:, 1:].flatten())
        
        # Radio físico en el percentil deseado
        umbral = float(np.percentile(distancias_vector, percentil_dist))
        radios_calculados[etiqueta_clase] = umbral
        
        # --- Lógica de conteo solicitada ---
        if etiqueta_clase in umbrales_manuales:
            u_manual = umbrales_manuales[etiqueta_clase]
            superan = np.sum(distancias_vector > u_manual)
            print(f"Clase {etiqueta_clase}: {superan} distancias superan el umbral de {u_manual}")
        # -----------------------------------
        
        color = cmap(i % 10)
        label_text = f'Clase {etiqueta_clase} (Radio: {umbral:.3f})'
        
        # Graficamos la línea
        linea, = plt.plot(np.linspace(0, 100, len(distancias_vector)), distancias_vector, 
                          color=color, linewidth=2)
        # Forzamos la vista para enfocarnos en los radios útiles (ignorando el percentil 100 extremo)
        # Como tu radio máximo de interés es ~5.03 (Clase 2), un límite de 15 es suficiente para todos.
        # if nombre_dataset=='shuttle':
        #     plt.ylim(0, 6)

        plt.scatter(percentil_dist, umbral, color=color, s=40, zorder=5)

        # Guardamos la info para la leyenda: (valor_radio, handle, etiqueta)
        legend_info.append((umbral, linea, label_text))

    # --- LÓGICA DE ORDENAMIENTO DE LEYENDA ---
    # Ordenamos de menor a mayor radio
    legend_info.sort(key=lambda x: x[0], reverse=False)
    
    handles_ordenados = [item[1] for item in legend_info]
    labels_ordenados = [item[2] for item in legend_info]
    # -----------------------------------------
       
    plt.axvline(percentil_dist, color='gray', linestyle='--', alpha=0.5, label=f'P{percentil_dist}')

    if nombre_dataset in nombre_configuracion_dataset:
        nombre_configuracion = nombre_configuracion_dataset[nombre_dataset]
        plt.title(f'Percentil Radio Distancia {percentil_dist} \n {nombre_dataset.upper()} \n {nombre_configuracion}', fontsize=13)
    else:
        plt.title(f'Percentil Radio Distancia {percentil_dist} \n {nombre_dataset.upper()}', fontsize=13) 

    plt.xlabel('Percentil (%)')
    plt.ylabel('Distancia Euclídea Escalada (Radio $u$)')
    
    # Aplicamos la leyenda ordenada
    plt.legend(
        handles_ordenados, 
        labels_ordenados, 
        loc='upper left', 
        bbox_to_anchor=(0.02, 0.98), 
        title="Clases",
        frameon=True,
        shadow=True
    )
    
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.tight_layout()

    if guardar_en:
        Path(guardar_en).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(guardar_en, bbox_inches="tight")
        plt.close()
    
    return radios_calculados

✅ Clase IsolationCleaner cargada correctamente.


In [7]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
from sklearn.metrics import balanced_accuracy_score, f1_score

from config_datasets import config_datasets
from cargar_dataset import cargar_dataset, graficar_distribucion_clases


# ======================================================================================
# CONFIG
# ======================================================================================
K_VECINOS_LISTA = [7]
N_SPLITS_CV = 5
RANDOM_STATE = 42
SIGMA_RUIDO = 0.01

# Crear carpetas si no existen
Path("resultados").mkdir(exist_ok=True)
Path("figuras").mkdir(exist_ok=True)


# ======================================================================================
# MÉTRICAS
# ======================================================================================
def macro_f1(y_true, y_pred) -> float:
    return float(f1_score(y_true, y_pred, average="macro"))


def balanced_acc(y_true, y_pred) -> float:
    return float(balanced_accuracy_score(y_true, y_pred))


# ======================================================================================
# ESTUDIO 1: Consistencia vecinal (pureza de vecinos de misma clase)
# ======================================================================================
def estudio_consistencia_vecinal(X: np.ndarray, y: np.ndarray, k: int) -> dict:
    nn = NearestNeighbors(n_neighbors=k + 1, metric="euclidean")
    nn.fit(X)
    idx_vecinos = nn.kneighbors(X, return_distance=False)[:, 1:]  # quitamos el propio punto

    consistencias = []
    por_clase = {}

    clases = np.unique(y)
    for c in clases:
        por_clase[c] = []

    for i in range(X.shape[0]):
        y_i = y[i]
        vecinos = idx_vecinos[i]

        misma = 0
        for j in vecinos:
            if y[j] == y_i:
                misma += 1

        c_i = misma / float(k)
        consistencias.append(c_i)
        por_clase[y_i].append(c_i)

    media_global = float(np.mean(consistencias)) if len(consistencias) > 0 else 0.0

    medias_clase = {}
    for c in clases:
        vals = por_clase[c]
        medias_clase[c] = float(np.mean(vals)) if len(vals) > 0 else 0.0

    return {
        "consistencia_global_media": media_global,
        "consistencia_media_por_clase": medias_clase,
    }


# ======================================================================================
# ESTUDIO 2: kNN baseline (CV)
# ======================================================================================
def estudio_knn_baseline_cv(X: np.ndarray, y: np.ndarray, k: int, n_splits: int, random_state: int) -> dict:
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    bacs = []
    f1s = []

    for train_idx, test_idx in skf.split(X, y):
        X_train = X[train_idx]
        y_train = y[train_idx]
        X_test = X[test_idx]
        y_test = y[test_idx]

        clf = KNeighborsClassifier(n_neighbors=k, metric="euclidean")
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        bacs.append(balanced_acc(y_test, y_pred))
        f1s.append(macro_f1(y_test, y_pred))

    return {
        "knn_bac_media": float(np.mean(bacs)) if len(bacs) > 0 else 0.0,
        "knn_bac_std": float(np.std(bacs, ddof=1)) if len(bacs) > 1 else 0.0,
        "knn_f1_macro_media": float(np.mean(f1s)) if len(f1s) > 0 else 0.0,
        "knn_f1_macro_std": float(np.std(f1s, ddof=1)) if len(f1s) > 1 else 0.0,
    }


# ======================================================================================
# ESTUDIO 3: Estabilidad de vecindario (Jaccard A vs B)
# B = X + ruido gaussiano leve
# ======================================================================================
def estudio_estabilidad_vecindario_jaccard(X: np.ndarray, k: int, sigma_ruido: float, random_state: int) -> dict:
    rng = np.random.default_rng(random_state)

    X_a = X
    ruido = rng.normal(loc=0.0, scale=sigma_ruido, size=X.shape)
    X_b = X + ruido

    nn_a = NearestNeighbors(n_neighbors=k + 1, metric="euclidean")
    nn_b = NearestNeighbors(n_neighbors=k + 1, metric="euclidean")
    nn_a.fit(X_a)
    nn_b.fit(X_b)

    idx_a = nn_a.kneighbors(X_a, return_distance=False)[:, 1:]
    idx_b = nn_b.kneighbors(X_b, return_distance=False)[:, 1:]

    jaccards = []

    for i in range(X.shape[0]):
        set_a = set(idx_a[i].tolist())
        set_b = set(idx_b[i].tolist())

        inter = len(set_a.intersection(set_b))
        uni = len(set_a.union(set_b))
        j = inter / float(uni) if uni > 0 else 0.0
        jaccards.append(j)

    return {
        "jaccard_media": float(np.mean(jaccards)) if len(jaccards) > 0 else 0.0,
        "jaccard_std": float(np.std(jaccards, ddof=1)) if len(jaccards) > 1 else 0.0,
        "sigma_ruido": sigma_ruido,
    }


# ======================================================================================
# ESTUDIO 4: Margen local (inter/intra)
# ======================================================================================
def estudio_margen_local(X: np.ndarray, y: np.ndarray, k: int, eps: float = 1e-9) -> dict:
    nn = NearestNeighbors(n_neighbors=k + 1, metric="euclidean")
    nn.fit(X)
    distancias, indices = nn.kneighbors(X, return_distance=True)

    distancias = distancias[:, 1:]
    indices = indices[:, 1:]

    ratios = []
    por_clase = {}

    clases = np.unique(y)
    for c in clases:
        por_clase[c] = []

    for i in range(X.shape[0]):
        y_i = y[i]
        idx_vec = indices[i]
        dist_vec = distancias[i]

        intra = []
        inter = []

        for pos in range(len(idx_vec)):
            j = idx_vec[pos]
            d = dist_vec[pos]
            if y[j] == y_i:
                intra.append(d)
            else:
                inter.append(d)

        if len(intra) == 0 or len(inter) == 0:
            continue

        d_intra = float(np.mean(intra))
        d_inter = float(np.mean(inter))
        r_i = d_inter / (d_intra + eps)

        ratios.append(r_i)
        por_clase[y_i].append(r_i)

    media_global = float(np.mean(ratios)) if len(ratios) > 0 else 0.0

    medias_clase = {}
    for c in clases:
        vals = por_clase[c]
        medias_clase[c] = float(np.mean(vals)) if len(vals) > 0 else 0.0

    return {
        "margen_local_media_global": media_global,
        "margen_local_media_por_clase": medias_clase,
        "n_validos": int(len(ratios)),
        "n_total": int(X.shape[0]),
    }


# ======================================================================================
# MAIN
# ======================================================================================
timestamp = datetime.now().strftime("%Y-%m-%d_%H%M")

nombre_archivo_txt = f"resultados/reporte_distribucion_y_vecindad_{timestamp}.txt"
nombre_archivo_csv = f"resultados/diagnostico_vecindad_{timestamp}.csv"

lineas_resultado = []
filas_csv = []

for nombre, cfg in config_datasets.items():
    lineas_resultado.append(f"\n🔍 Analizando dataset: {nombre.upper()}")
    print(f"\n🔍 Analizando dataset: {nombre.upper()}")

    try:
        names = cfg.get("esquema") if cfg.get("header", None) is None else None

        if nombre not in ["shuttle"]:
            continue

        X, y, _ = cargar_dataset(
            path=cfg.get("path"),
            clase_minoria=cfg.get("clase_minoria"),
            col_features=cfg.get("col_features"),
            col_target=cfg.get("col_target"),
            sep=cfg.get("sep", ","),
            header=cfg.get("header", None),
            binarizar=False,
            tipo=cfg.get("tipo", "tabular"),
            impute="median",
            names=names
        )



        if isinstance(X, pd.DataFrame):
            X = X.to_numpy()
        else:
            X = np.asarray(X)

        y = np.asarray(y)

        # -------------------------
        # Distribución de clases
        # -------------------------
        conteo = pd.Series(y).value_counts()
        clase_min_real = conteo.idxmin()
        total = conteo.sum()
        proporcion = (conteo / total * 100).round(2)

        print("🎯 Valores únicos del target:", list(conteo.index))
        print("📊 Distribución de clases:")

        lineas_resultado.append(f"🎯 Valores únicos del target: {list(conteo.index)}")
        lineas_resultado.append("📊 Distribución de clases:")

        for clase, count in conteo.items():
            print(f"   - {clase}: {count} ({proporcion[clase]}%)")
            lineas_resultado.append(f"   - {clase}: {count} ({proporcion[clase]}%)")

        lineas_resultado.append(f"✅ Clase minoritaria real: {clase_min_real}")
        lineas_resultado.append(f"⚠️ Clase configurada como minoritaria: {cfg.get('clase_minoria')}")

        # Gráfico de distribución
        nombre_figura = f"figuras/{nombre.lower()}_distribucion_{timestamp}.png"
        graficar_distribucion_clases(y, nombre_dataset=nombre, guardar_en=nombre_figura)


        nombre_configuracion_dataset = {
            "shuttle" : "PRD70_PR30_CPprop_UD025_Upp040_UR030_I1",
            "us_crime" : "PRD85_PR40_CPent_UD045_PE80_UR045_I0"
        }

        percentil = 85
        porcentaje_limpieza = 0
        nombre_figura_comp = f"figuras/percentiles/{nombre.lower()}_comparativa_clases_{timestamp}.png"

        # Usamos la función que integra IsolationCleaner
        dict_radios = graficar_percentiles_limpios_clase_custom(
            X, y, 
            k=7, 
            percentil_dist=percentil,      # Alineado con el parámetro de la función
            porcentaje_limpieza=porcentaje_limpieza,       # Configuración de limpieza del 1%
            test_size=0.2,                 # Split definido en notebook
            random_state=42,               # Consistente con el proceso base
            nombre_dataset=nombre, 
            guardar_en=nombre_figura_comp,
            nombre_configuracion_dataset=nombre_configuracion_dataset
        )

        print(f"   📊 Radios P{percentil} por clase en {nombre} (con limpieza IF por clase): {dict_radios}")

        # -------------------------
        # Diagnóstico de vecindad (4 estudios)
        # -------------------------
        lineas_resultado.append("🧭 Diagnóstico de vecindad (métrica euclídea sobre X cargado):")

        for k in K_VECINOS_LISTA:
            r1 = estudio_consistencia_vecinal(X, y, k)
            r2 = estudio_knn_baseline_cv(X, y, k, N_SPLITS_CV, RANDOM_STATE)
            r3 = estudio_estabilidad_vecindario_jaccard(X, k, SIGMA_RUIDO, RANDOM_STATE)
            r4 = estudio_margen_local(X, y, k)

            linea = (
                f"   k={k:<2} | "
                f"consistencia={r1['consistencia_global_media']:.4f} | "
                f"kNN_BAC={r2['knn_bac_media']:.4f}±{r2['knn_bac_std']:.4f} | "
                f"kNN_F1m={r2['knn_f1_macro_media']:.4f}±{r2['knn_f1_macro_std']:.4f} | "
                f"Jaccard={r3['jaccard_media']:.4f}±{r3['jaccard_std']:.4f} | "
                f"margen={r4['margen_local_media_global']:.4f} (validos={r4['n_validos']}/{r4['n_total']})"
            )
            print(linea)
            lineas_resultado.append(linea)

            filas_csv.append({
                "dataset": nombre,
                "k": k,
                "n_muestras": int(X.shape[0]),
                "n_features": int(X.shape[1]),

                "consistencia_vecinal_media": float(r1["consistencia_global_media"]),
                "knn_bac_media": float(r2["knn_bac_media"]),
                "knn_bac_std": float(r2["knn_bac_std"]),
                "knn_f1_macro_media": float(r2["knn_f1_macro_media"]),
                "knn_f1_macro_std": float(r2["knn_f1_macro_std"]),
                "jaccard_media": float(r3["jaccard_media"]),
                "jaccard_std": float(r3["jaccard_std"]),
                "margen_local_media": float(r4["margen_local_media_global"]),
                "margen_local_n_validos": int(r4["n_validos"]),
                "margen_local_n_total": int(r4["n_total"]),
            })

    except Exception as e:
        print(f"❌ Error al analizar {nombre}: {e}")
        lineas_resultado.append(f"❌ Error al analizar {nombre}: {e}")

# Guardar TXT
with open(nombre_archivo_txt, "w", encoding="utf-8") as f:
    f.write("\n".join(lineas_resultado))

# Guardar CSV
df_out = pd.DataFrame(filas_csv)
df_out.to_csv(nombre_archivo_csv, index=False, encoding="utf-8")

print(f"\n📁 Reporte guardado en: {nombre_archivo_txt}")
print(f"📁 CSV guardado en: {nombre_archivo_csv}")



🔍 Analizando dataset: US_CRIME

🔍 Analizando dataset: SHUTTLE
🎯 Valores únicos del target: [1, 4, 5, 3, 2, 7, 6]
📊 Distribución de clases:
   - 1: 45586 (78.6%)
   - 4: 8903 (15.35%)
   - 5: 3267 (5.63%)
   - 3: 171 (0.29%)
   - 2: 50 (0.09%)
   - 7: 13 (0.02%)
   - 6: 10 (0.02%)

--- TAMAÑO DE LOS CONJUNTOS (SHUTTLE) ---
Total de datos originales: 58000
Tamaño Train: 46400 muestras (80%)
Tamaño Test:  11600 muestras (20%)

--- DISTRIBUCIÓN DE CLASES EN TRAIN (80% de los datos) ---
Clase 1: 36469 instancias
Clase 2: 40 instancias
Clase 3: 137 instancias
Clase 4: 7122 instancias
Clase 5: 2614 instancias
Clase 6: 8 instancias
Clase 7: 10 instancias
Total Train: 46400
🧹 IF (por_clase, percentil=0.0%): removidos 0; quedan 46400 de 46400.
Clase 1: 5724 distancias superan el umbral de 1.366
Clase 2: 6 distancias superan el umbral de 22.34
Clase 3: 33 distancias superan el umbral de 8.502
Clase 4: 1212 distancias superan el umbral de 1.958
Clase 5: 504 distancias superan el umbral de 4.118
C

In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# -------------------------
# CONFIG
# -------------------------
RUTA_CSV = "resultados/diagnostico_vecindad_2026-01-01_1002.csv"  
K_ELEGIDO = 7
SALIDA_DIR = Path("figuras_radar")
SALIDA_DIR.mkdir(exist_ok=True)

# Normalización del margen:
# - ratio ~1: malo (inter ~ intra)
# - ratio >1: mejor separación local
# Cap explícito para meterlo en [0,1] sin distorsión.
MARGEN_CAP = 5.0  # si querés ser más estricto: 2.0

# -------------------------
# CARGA + FILTRO
# -------------------------
df = pd.read_csv(RUTA_CSV)

# Nos quedamos con un k fijo (recomendado) para que el radar sea comparable
dfk = df[df["k"] == K_ELEGIDO].copy()

# Si hay datasets repetidos por cualquier razón, consolidamos por media
agr = dfk.groupby("dataset", as_index=False).agg({
    "knn_bac_media": "mean",
    "knn_f1_macro_media": "mean",
    "consistencia_vecinal_media": "mean",
    "margen_local_media": "mean"
})

# -------------------------
# NORMALIZACIÓN (sin jaccard)
# -------------------------
# BAC, F1m y consistencia ya están en [0,1]
# Margen: normalización anclada en 1 (punto neutro) y escalada por el máximo observado

# -------------------------
# NORMALIZACIÓN LIMPIA DEL MARGEN (r / (1 + r))
# r = d_inter / d_intra
# - r = 1  -> 0.5 (zona neutra, solapamiento)
# - r > 1  -> >0.5 (separación)
# - r < 1  -> <0.5 (vecindad mala)
# -------------------------
margen_norm = []

for v in agr["margen_local_media"].tolist():
    if pd.isna(v) or v <= 0:
        margen_norm.append(0.0)
    else:
        margen_norm.append(float(v) / (1.0 + float(v)))

agr["margen_norm"] = margen_norm


# -------------------------
# RADAR POR DATASET (matplotlib puro)
# -------------------------
labels = ["BAC", "F1m", "Consistencia", "Separación local (norm.)"]
n_vars = len(labels)

# ángulos del radar
angulos = np.linspace(0, 2*np.pi, n_vars, endpoint=False).tolist()
angulos += angulos[:1]  # cerrar

for i in range(agr.shape[0]):
    nombre = str(agr.loc[i, "dataset"])

    bac = float(agr.loc[i, "knn_bac_media"])
    f1m = float(agr.loc[i, "knn_f1_macro_media"])
    cons = float(agr.loc[i, "consistencia_vecinal_media"])
    marg = float(agr.loc[i, "margen_norm"])

    valores = [bac, f1m, cons, marg]
    valores += valores[:1]  # cerrar

    fig = plt.figure(figsize=(7, 7))
    ax = plt.subplot(111, polar=True)

    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)

    ax.set_xticks(angulos[:-1])
    ax.set_xticklabels(labels)

    ax.tick_params(axis="x", pad=8)

    ax.set_ylim(0, 1.1)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(["0.2", "0.4", "0.6", "0.8", "1.0"])


    ax.plot(angulos, valores, linewidth=2)
    ax.fill(angulos, valores, alpha=0.15)

    margen_crudo = float(agr.loc[i, "margen_local_media"])

    titulo_linea_1 = f"{nombre} | k={K_ELEGIDO}"
    titulo_linea_2 = f"ratio inter/intra={margen_crudo:.3f} | separación_norm={marg:.2f}"

    ax.set_title(titulo_linea_1 + "\n" + titulo_linea_2, y=1.12, pad=0)

    out = SALIDA_DIR / f"radar_{nombre}_k{K_ELEGIDO}.png"

    plt.subplots_adjust(top=0.90)
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)

print(f"Listo. Radares guardados en: {SALIDA_DIR.resolve()}")


Listo. Radares guardados en: C:\Users\FamiliaNatelloMedina\Desktop\codigo\datasets\figuras_radar
